# Scraping the trailer bills
This file provides to scrape the trailer bills.

In [45]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Test by using SB-105
SB-105 is "Budget Acts of 2022 and 2023" as a trailer bill.

In [46]:
# -------------------------
# Fetch and clean bill text
# -------------------------
def fetch_text(url):
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    return soup.get_text("\n")

# -------------------------
# Patterns
# -------------------------
amount_only_line_pattern = re.compile(r"^[\(\−\-]?\$?\d[\d,]*[\)]?$")

agency_header_pattern = re.compile(r"^[A-Z][A-Z0-9\/\-\s]+$")
skip_agency_headers = re.compile(r"^(CHAPTER|SECTION|ARTICLE)\b", re.IGNORECASE)

item_pattern = re.compile(r"^(\d{4}-\d{3}-\d{4})[—\-]\s*(.+)$")
program_pattern = re.compile(r"^(\d{4})-(.+?)(?:\.{2,}|…)?$", re.IGNORECASE)
object_pattern = re.compile(r"^(\d{6,7})-(.+?)(?:\.{2,}|…)?$", re.IGNORECASE)

schedule_header_pattern = re.compile(r"^Schedule:", re.IGNORECASE)
schedule_group_pattern = re.compile(r"^\(([\d\.]+)\)\s*$")

dotleader_trailing_amount = re.compile(
    r"(?:\.{2,}|…)\s*\$?([\d]{1,3}(?:,\d{3})+|\d+)\s*$"
)

# To detect section headers like "SEC. 4. Item 3360-002-0890 is added to Section 2.00 of the Budget Act of 2023, to read:"
section_pattern = re.compile(r"^(SEC\.|SECTION)\s+\d+\.", re.IGNORECASE)
budget_act_year_pattern = re.compile(r"Budget\s+Act\s+of\s+(20\d{2})", re.IGNORECASE)

# To record whether this section is from an "added" or "amended" provision
action_pattern = re.compile(r"is\s+(added|amended|repealed)(?:\s+to\s+read)?", re.IGNORECASE)

# To recognize the source of the change (e.g. "as added by Senate Bill 104")
source_pattern = re.compile(r"as\s+(?:added|amended)\s+by\s+(Senate\s+Bill\s+\d+|Assembly\s+Bill\s+\d+)", re.IGNORECASE)

In [47]:
# -------------------------
# Helpers
# -------------------------
def clean_amount(s: str) -> int:
    return int(
        s.replace(",", "")
         .replace("(", "-")
         .replace(")", "")
         .replace("−", "-")
         .replace("$", "")
         .strip()
    )

def extract_dotleader_amount(line: str):
    m = dotleader_trailing_amount.search(line)
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def find_amount_for_line(lines, i, max_lookahead=6):
    """
    Returns (amount, next_index_to_continue_from).
    Priority:
      1) dot-leader amount at end of current line
      2) amount on next line by itself
      3) dot-leader amount within next few lines
    """
    amt = extract_dotleader_amount(lines[i])
    if amt is not None:
        return amt, i + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        if amount_only_line_pattern.match(lines[j]):
            return clean_amount(lines[j]), j + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        amt2 = extract_dotleader_amount(lines[j])
        if amt2 is not None:
            return amt2, j + 1

    return None, i + 1

def extract_fund_name_from_item_title(item_title: str):
    m = re.search(r"payable\s+from\s+the\s+(.+)$", item_title, flags=re.IGNORECASE)
    return m.group(1).strip().rstrip(".") if m else None

# -------------------------
# Monetary-only parser (NO ACTIVITY)
# -------------------------
def parse_budget_money_only(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    current_section_year = None # added section header detection to track year for items without explicit fund name
    current_agency = None
    current_item = None
    current_program_code = None
    current_fund_code = None
    current_fund_name = None

    in_schedule = False
    current_schedule_group = None

    rows = []
    i = 0

    while i < len(lines):
        line = lines[i]
        # Detect section headers to track current year for items without explicit fund name
        if section_pattern.match(line):
            sec_match = re.search(r'(\d+)', line)
            current_section_no = sec_match.group(1) if sec_match else None
            
            # Look ahead a bit further (10 lines) and combine
            header_text = " ".join(lines[i : i + 10])
            header_combined = header_text.lower()
            
            m_year = budget_act_year_pattern.search(header_text)
            current_section_year = m_year.group(1) if m_year else None

            # Action detection
            if "is added" in header_combined:
                current_action = "Added"
            elif "is repealed" in header_combined:
                current_action = "Repealed"
            elif "is amended" in header_combined:
                current_action = "Amended"
            else:
                current_action = "Unknown"
            
            source_match = source_pattern.search(header_text)
            current_source = source_match.group(1).replace("Senate Bill ", "SB").replace("Assembly Bill ", "AB") if source_match else "Original"
            
            i += 1
            continue

        # Agency header
        if (
            agency_header_pattern.match(line)
            and len(line) > 8
            and not skip_agency_headers.match(line)
        ):
            current_agency = line.strip()
            i += 1
            continue

        # Schedule header
        if schedule_header_pattern.match(line):
            in_schedule = True
            current_schedule_group = None
            i += 1
            continue

        # Schedule group like "(2)" or "(.5)"
        m_sg = schedule_group_pattern.match(line)
        if m_sg and in_schedule:
            current_schedule_group = m_sg.group(1)
            i += 1
            continue

        # Item
        m_item = item_pattern.match(line)
        if m_item:
            current_item = m_item.group(1)
            item_title = m_item.group(2).strip()
            
            # Reset sub-levels
            current_program_code = None
            in_schedule = False
            current_schedule_group = None
            current_fund_code = current_item.split("-")[-1]
            current_fund_name = extract_fund_name_from_item_title(item_title)

            amt, next_i = find_amount_for_line(lines, i)

            # --- MODIFIED: Added logic to capture Repealed even if no amount is found ---
            if amt is not None or current_action == "Repealed":
                final_amt = 0 if current_action == "Repealed" else amt
                rows.append({
                    "level": "item",
                    "budget_year": current_section_year,
                    "action_type": current_action,
                    "ref_source": current_source,
                    "agency": current_agency,
                    "item_number": current_item,
                    "program_code": None,
                    "object_code": None,
                    "fund_code": current_fund_code,
                    "fund_name": current_fund_name,
                    "schedule_group": None,
                    "name": item_title,
                    "amount": final_amt
                })
                # If it was repealed, amt might be None, so we just move to next line
                i = next_i if amt is not None else i + 1
                continue

        # Ignore everything before first item
        if current_item is None:
            i += 1
            continue

        # Program
        m_prog = program_pattern.match(line)
        if m_prog:
            current_program_code = m_prog.group(1)
            prog_name = m_prog.group(2).strip()

            amt, next_i = find_amount_for_line(lines, i)
            if amt is not None:
                final_amt = 0 if current_action == "Repealed" else amt
                rows.append({
                    "level": "item",
                    "budget_year": current_section_year,
                    "action_type": current_action,
                    "ref_source": current_source,
                    "agency": current_agency,
                    "item_number": current_item,
                    "program_code": current_program_code,
                    "object_code": None,
                    "fund_code": current_fund_code,
                    "fund_name": current_fund_name,
                    "schedule_group": current_schedule_group if in_schedule else None,
                    "name": prog_name,
                    "amount": final_amt
                })
                i = next_i
                continue

            i += 1
            continue

        # Object / Project
        m_obj = object_pattern.match(line)
        if m_obj:
            obj_code = m_obj.group(1)
            obj_name = m_obj.group(2).strip()

            amt, next_i = find_amount_for_line(lines, i)
            if amt is not None:
                program_code_for_row = current_program_code or obj_code[:4]

                final_amt = 0 if current_action == "Repealed" else amt

                rows.append({
                    "level": "object",
                    "budget_year": current_section_year,
                    "action_type": current_action,
                    "ref_source": current_source,
                    "agency": current_agency,
                    "item_number": current_item,
                    "program_code": program_code_for_row,
                    "object_code": obj_code,
                    "fund_code": current_fund_code,
                    "fund_name": current_fund_name,
                    "schedule_group": current_schedule_group if in_schedule else None,
                    "name": obj_name,
                    "amount": final_amt
                })

                i = next_i
                continue

            i += 1
            continue

        i += 1

    return rows


In [48]:
# -------------------------
# Run + save CSV for SB105
# -------------------------

SB105 = "https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=202320240SB105"

def main():
    print("Fetching SB105 text...")
    text = fetch_text(SB105)

    print("Parsing SB105 — MONEY ONLY...")
    data = parse_budget_money_only(text)
    df = pd.DataFrame(data)

    print("\nSample rows:")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows: {len(df)}")

    out = "2023_SB105_CA_Budget_money_only.csv"
    df.to_csv(out, index=False)
    print(f"Saved to: {out}")

if __name__ == "__main__":
    main()

Fetching SB105 text...
Parsing SB105 — MONEY ONLY...

Sample rows:
 level budget_year action_type ref_source agency   item_number program_code object_code fund_code                                           fund_name schedule_group                                                                                                                             name     amount
  item        2022     Amended   Original   None 6100-149-0890         None        None      0890                                                None           None  For local assistance, State Department of Education, American Rescue Plan Act for After School and Child Care Programs, payable 3324616000
object        2022     Amended   Original   None 6100-149-0890         5210     5210048      0890                                                None              1                                                                                                            After School Programs   30710000
object        2022

## Run regarding AB 102 and SB 104

In [49]:
AB102 = "https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=202320240AB102"
SB104 = "https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=202320240SB104"

In [50]:
def main():
    print("Fetching AB102 text...")
    text = fetch_text(AB102)

    print("Parsing AB102 — MONEY ONLY...")
    data = parse_budget_money_only(text)
    df = pd.DataFrame(data)

    print("\nSample rows:")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows: {len(df)}")

    out = "2023_AB102_CA_Budget_money_only.csv"
    df.to_csv(out, index=False)
    print(f"Saved to: {out}")

if __name__ == "__main__":
    main()

Fetching AB102 text...
Parsing AB102 — MONEY ONLY...

Sample rows:
 level budget_year action_type ref_source agency   item_number program_code object_code fund_code                           fund_name schedule_group                                                                                                                             name     amount
  item        2023     Amended   Original   None 0250-001-0001         None        None      0001                                None           None                                                                                                   For support of Judicial Branch  620021000
  item        2023     Amended   Original   None 0250-001-0001         0130        None      0001                                None              1                                                                                                                    Supreme Court   55790000
  item        2023     Amended   Original   None 0250-001-0001    

In [51]:
def main():
    print("Fetching SB104 text...")
    text = fetch_text(SB104)

    print("Parsing SB104 — MONEY ONLY...")
    data = parse_budget_money_only(text)
    df = pd.DataFrame(data)

    print("\nSample rows:")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows: {len(df)}")

    out = "2023_SB104_CA_Budget_money_only.csv"
    df.to_csv(out, index=False)
    print(f"Saved to: {out}")

if __name__ == "__main__":
    main()

Fetching SB104 text...
Parsing SB104 — MONEY ONLY...

Sample rows:
 level budget_year action_type ref_source          agency   item_number program_code object_code fund_code                     fund_name schedule_group                                                                                                                                      name     amount
  item        2023     Amended   Original AND FOOD ACCESS 0250-101-0001         None        None      0001                          None           None                                                                                                     For local assistance, Judicial Branch  140473000
object        2023     Amended   Original AND FOOD ACCESS 0250-101-0001         0150     0150010      0001                          None              1                                                                                                     Support for Operation of Trial Courts   77501000
object        2023     Amended